In [ ]:
# @title 🛠️ Setup & Security Configuration
# @markdown Run this cell to initialize the environment and load secrets.

import os
import torch
import gc
from google.colab import userdata
from IPython.display import clear_output

# --- Security & API Keys ---
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    CIVITAI_API_KEY = userdata.get('CIVITAI_API_TOKEN')
    NGROK_TOKEN = userdata.get('Ngrok')
except:
    HF_TOKEN = None
    CIVITAI_API_KEY = None
    NGROK_TOKEN = None
    print("⚠️ Warning: Secrets not found. Some models may not download.")

# --- Memory Management ---
def flush():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

# --- Installation ---
print("Installing dependencies... This may take a minute.")
!pip install -q diffusers transformers accelerate bitsandbytes gradio omegaconf safetensors
!pip install -q git+https://github.com/huggingface/peft.git

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

clear_output()
print(f"✅ Environment Ready. Running on: {DEVICE}")

✅ Environment Ready. Running on: cuda


In [ ]:
from diffusers import DiffusionPipeline, AutoencoderKL, StableDiffusionXLPipeline
from diffusers import DPMSolverMultistepScheduler
import requests
import torch

class ModelManager:
    def __init__(self):
        self.pipe = None
        self.current_model = ""

    def load_hf_model(self, model_id):
        print(f"Loading HF Model: {model_id}...")
        # Ensure DEVICE and torch_dtype are available
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32

        self.pipe = DiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=dtype,
            use_safetensors=True,
            token=globals().get('HF_TOKEN')
        ).to(device)

        # Enable VAE slicing and tiling for reduced VRAM
        if self.pipe.vae:
            self.pipe.vae.enable_slicing()
            self.pipe.vae.enable_tiling()

        # Enable attention slicing for further VRAM reduction
        self.pipe.enable_attention_slicing()

        if device == "cuda":
            self.pipe.enable_model_cpu_offload()

        self.current_model = model_id
        return f"Successfully loaded {model_id}"

    def download_civitai_model(self, url, filename="model.safetensors"):
        print("Downloading from Civitai...")
        headers = {"Authorization": f"Bearer {globals().get('CIVITAI_API_KEY')}"} if globals().get('CIVITAI_API_KEY') else {}
        response = requests.get(url, headers=headers, stream=True)
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return filename

manager = ModelManager()

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
import gradio as gr
from diffusers import DiffusionPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from transformers import T5EncoderModel, BitsAndBytesConfig
import torch
from google.colab import userdata

class OptimizedEngine:
    def __init__(self):
        self.pipe = None

    def load_model(self, model_id, use_4bit=True):
        flush()
        yield "🔄 Cleaning memory and initializing..."

        try:
            # Try to get token, handle cases where access isn't granted
            try:
                token = userdata.get('HF_TOKEN')
            except Exception:
                token = None
                yield "⚠️ Warning: HF_TOKEN access not granted. Please check the Secrets tab (🔑) and enable 'Notebook access'."

            # 1. Configure 4-bit quantization
            nf4_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            ) if use_4bit else None

            yield f"⏳ Loading {model_id}..."

            # 2. Load Pipeline
            self.pipe = DiffusionPipeline.from_pretrained(
                model_id,
                torch_dtype=torch.float16,
                quantization_config=nf4_config,
                device_map="cuda",
                token=token
            )

            # 3. Efficiency Hooks
            if hasattr(self.pipe, "vae") and self.pipe.vae is not None:
                self.pipe.enable_vae_tiling()
                self.pipe.enable_vae_slicing()

            self.pipe.enable_sequential_cpu_offload()

            yield f"✅ Ready! Model loaded successfully."
        except Exception as e:
            yield f"❌ Error: {str(e)}"

engine = OptimizedEngine()

# @title 🎨 Launch Secure Optimized UI
USER = "admin" # @param {type:"string"}
PASS = "2025" # @param {type:"string"}

MODELS = [
    "Wan-AI/Wan2.1-T2I-1.3B",
    "stabilityai/stable-diffusion-xl-base-1.0",
    "stabilityai/sdxl-turbo",
    "Lykon/DreamShaper-XL-v2-Turbo",
    "Custom (Use Box Below)"
]

def generate(prompt, n_prompt, steps, cfg, w, h, seed):
    if engine.pipe is None: return None, "Please load a model first!"

    flush()
    generator = torch.Generator("cuda").manual_seed(int(seed)) if seed != -1 else None

    try:
        image = engine.pipe(
            prompt=prompt,
            negative_prompt=n_prompt,
            num_inference_steps=int(steps),
            guidance_scale=cfg,
            width=int(w),
            height=int(h),
            generator=generator
        ).images[0]

        flush()
        return image, "Generation Successful"
    except Exception as e:
        return None, f"Error: {str(e)}"

with gr.Blocks() as ui:
    gr.Markdown("## ⚡ Optimized Private Studio (Colab Free Edition)")

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### 1. Resource Setup")
                m_select = gr.Dropdown(MODELS, label="Select Model", value=MODELS[0])
                m_custom = gr.Textbox(label="OR: Custom HF ID", placeholder="author/model-name")
                q_bit = gr.Checkbox(label="Enable 4-Bit Quantization", value=True)
                load_btn = gr.Button("Initialize Engine", variant="primary")
                log_box = gr.Textbox(label="Status", interactive=False)

            with gr.Group():
                gr.Markdown("### 2. Generation")
                prompt = gr.TextArea(label="Prompt", lines=3)
                n_prompt = gr.TextArea(label="Negative", value="blur, low quality", lines=1)
                with gr.Row():
                    steps = gr.Slider(1, 50, value=25, step=1, label="Steps")
                    cfg = gr.Slider(1, 15, value=7, step=0.5, label="CFG")
                with gr.Row():
                    width = gr.Slider(512, 1024, value=832, step=64, label="Width")
                    height = gr.Slider(512, 1024, value=832, step=64, label="Height")
                seed = gr.Number(label="Seed (-1 = Random)", value=-1)
                gen_btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            out_img = gr.Image(label="Output Image", type="pil")
            out_info = gr.Textbox(label="Process Info")
            clear_mem = gr.Button("♻️ Emergency RAM Clear")

    def load_wrapper(m_s, m_c, q):
        target = m_c if m_s == "Custom (Use Box Below)" else m_s
        yield from engine.load_model(target, q)

    load_btn.click(load_wrapper, [m_select, m_custom, q_bit], log_box)
    gen_btn.click(generate, [prompt, n_prompt, steps, cfg, width, height, seed], [out_img, out_info])
    clear_mem.click(lambda: flush(), None, out_info)

ui.launch(share=True, auth=(USER, PASS))

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20939109391b88620c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# @title 🔄 Quick Model Switcher
# @markdown Enter a Hugging Face Model ID to swap the current model.

# @markdown <em>**Only Use if issue with the previous cell get stuck or unresponsive.**</em>

NEW_MODEL_ID = "runwayml/stable-diffusion-v1-5" # @param {type:"string"}

try:
    # Check if manager exists, if not, try to alert the user
    if 'manager' not in globals():
        print("❌ Error: 'manager' is not defined. Please run the '🏗️ Model Engine & Downloader' cell first.")
    else:
        result = manager.load_hf_model(NEW_MODEL_ID)
        print(f"✅ {result}")
except Exception as e:
    print(f"❌ Failed to load model: {e}")